# Cross-Sell Basket Features

Bu notebook `data/gold/baskets.parquet` dosyasını okur, tarih bazlı train/test ayrımı yapar ve cross-sell modeli için temel ürün/sepet feature dosyalarını üretir.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Project root could not be found from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / "data" / "gold" / "baskets.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data" / "gold"
METRICS_DIR = PROJECT_ROOT / "artifacts" / "metrics"

TRAIN_PATH = OUTPUT_DIR / "cross_sell_train_baskets.parquet"
TEST_PATH = OUTPUT_DIR / "cross_sell_test_baskets.parquet"
ITEM_FEATURES_PATH = OUTPUT_DIR / "cross_sell_item_features.parquet"
SPLIT_SUMMARY_PATH = METRICS_DIR / "cross_sell_split_summary.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

## Parameters

In [ ]:
# Son 7 günü test seti olarak ayırıyoruz. Veri dönemine göre değiştirebilirsin.
TEST_DAYS = 7

# Cross-sell için tek ürünlü sepetler eğitim sinyali taşımaz.
MIN_BASKET_SIZE = 2

# Çok büyük sepetler pair üretirken gürültü ve maliyet yaratabilir.
MAX_BASKET_SIZE_FOR_MODELING = 50

## Load And Clean Baskets

In [ ]:
def normalize_articles(value):
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, set)):
        return [str(article) for article in value if pd.notna(article)]
    return [str(value)]


baskets = pd.read_parquet(DATA_PATH)
baskets["t_dat"] = pd.to_datetime(baskets["t_dat"])
baskets["articles"] = baskets["articles"].apply(normalize_articles)
baskets["basket_size"] = baskets["articles"].apply(len).astype("int32")

baskets = baskets.dropna(subset=["t_dat", "customer_id"])
baskets = baskets[baskets["basket_size"] >= MIN_BASKET_SIZE].copy()
baskets = baskets[baskets["basket_size"] <= MAX_BASKET_SIZE_FOR_MODELING].copy()
baskets = baskets.sort_values("t_dat").reset_index(drop=True)

baskets.head()

In [ ]:
baskets_summary = {
    "rows": int(len(baskets)),
    "unique_customers": int(baskets["customer_id"].nunique()),
    "min_date": baskets["t_dat"].min().strftime("%Y-%m-%d"),
    "max_date": baskets["t_dat"].max().strftime("%Y-%m-%d"),
    "mean_basket_size": float(baskets["basket_size"].mean()),
    "median_basket_size": float(baskets["basket_size"].median()),
}

baskets_summary

## Time-Based Train/Test Split

In [ ]:
max_date = baskets["t_dat"].max()
cutoff_date = max_date - pd.Timedelta(days=TEST_DAYS)

train_baskets = baskets[baskets["t_dat"] <= cutoff_date].copy()
test_baskets = baskets[baskets["t_dat"] > cutoff_date].copy()

if test_baskets.empty:
    fallback_cutoff = baskets["t_dat"].quantile(0.8)
    train_baskets = baskets[baskets["t_dat"] <= fallback_cutoff].copy()
    test_baskets = baskets[baskets["t_dat"] > fallback_cutoff].copy()
    cutoff_date = fallback_cutoff

split_summary = {
    **baskets_summary,
    "split_type": "time_based",
    "test_days": TEST_DAYS,
    "cutoff_date": pd.Timestamp(cutoff_date).strftime("%Y-%m-%d"),
    "train_rows": int(len(train_baskets)),
    "test_rows": int(len(test_baskets)),
    "train_min_date": train_baskets["t_dat"].min().strftime("%Y-%m-%d"),
    "train_max_date": train_baskets["t_dat"].max().strftime("%Y-%m-%d"),
    "test_min_date": test_baskets["t_dat"].min().strftime("%Y-%m-%d"),
    "test_max_date": test_baskets["t_dat"].max().strftime("%Y-%m-%d"),
}

split_summary

## Item-Level Features

In [ ]:
train_items = (
    train_baskets[["t_dat", "customer_id", "articles"]]
    .explode("articles")
    .rename(columns={"articles": "article_id"})
)

item_features = (
    train_items.groupby("article_id")
    .agg(
        item_basket_count=("article_id", "size"),
        unique_customer_count=("customer_id", "nunique"),
        first_seen_date=("t_dat", "min"),
        last_seen_date=("t_dat", "max"),
    )
    .reset_index()
)

last_train_date = train_baskets["t_dat"].max()
item_features["days_since_last_seen"] = (
    last_train_date - item_features["last_seen_date"]
).dt.days.astype("int32")
item_features["item_support"] = item_features["item_basket_count"] / len(train_baskets)
item_features["recency_score"] = 1 / (1 + item_features["days_since_last_seen"])
item_features = item_features.sort_values("item_basket_count", ascending=False).reset_index(drop=True)

item_features.head(10)

## Save Outputs

In [ ]:
train_baskets.to_parquet(TRAIN_PATH, index=False)
test_baskets.to_parquet(TEST_PATH, index=False)
item_features.to_parquet(ITEM_FEATURES_PATH, index=False)

with SPLIT_SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(split_summary, f, indent=2)

print(f"Saved train: {TRAIN_PATH}")
print(f"Saved test: {TEST_PATH}")
print(f"Saved item features: {ITEM_FEATURES_PATH}")
print(f"Saved split summary: {SPLIT_SUMMARY_PATH}")